# Lab Day 2 — Backbone, công thức huấn luyện và suy luận trên DeepWeeds

Notebook **khung** cho Colab hoặc Kaggle. Các ô có chữ `TODO` là phần **bạn tự viết**; ô cài đặt, tải dữ liệu
và chạy `eval.py` đã viết sẵn. Đọc `README.md` (quy tắc chia dữ liệu, cách đánh giá), `GUIDE.md` và `RUBRIC.md` trước.

**Quy tắc quan trọng nhất:** chọn mọi thứ trên **val**; **test chỉ chạy một lần mỗi seed ở Bước 4**.

Bản đồ: Bước 0 (EDA, kiểm tra) → Bước 1 (backbone) → Bước 2 (công thức huấn luyện) → Bước 3 (suy luận)
→ Bước 4 (chung kết + `eval.py`) → Bước 5 (xlsx, biểu đồ, báo cáo).

## 0. Cài đặt môi trường

In [ ]:
# Cài thư viện (Colab/Kaggle). Kaggle: bật Internet trong Settings.
!pip -q install timm openpyxl

import sys, os, platform
import numpy as np, pandas as pd, torch, timm

# Clone fork vào /content (Colab). Đổi REPO_DIR nếu bạn đặt chỗ khác.
REPO_DIR = "K4-D02-NguyenVanQuocViet-2A202602973"
if not os.path.isdir(REPO_DIR):
    !git clone -q https://github.com/vietvuivui/{REPO_DIR}.git
CODE_DIR = f"{REPO_DIR}/submissions/2A202602973_NguyenVanQuocViet/code"
sys.path.insert(0, REPO_DIR)                 # để import eval.py
sys.path.insert(0, CODE_DIR)                 # code/ của bạn (đã chép từ starter/)

print("python", platform.python_version(), "| torch", torch.__version__, "| timm", timm.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ GPU")

### Tải dữ liệu (đã viết sẵn)
Ảnh từ Zenodo (~490 MB, kiểm tra MD5), nhãn và các file chia fold từ GitHub của tác giả.
**Sau khi giải nén, hãy `ls` để xem ảnh nằm ở thư mục nào** rồi đặt `IMAGES_DIR` cho đúng.

In [ ]:
import hashlib

os.makedirs("data/labels", exist_ok=True)
if not os.path.exists("data/images.zip"):
    !wget -q -O data/images.zip "https://zenodo.org/records/7939060/files/images.zip?download=1"

h = hashlib.md5()
with open("data/images.zip", "rb") as f:
    for chunk in iter(lambda: f.read(1 << 20), b""):
        h.update(chunk)
assert h.hexdigest() == "b7b30f96d466fba86016aa5a26606e0f", f"MD5 sai: {h.hexdigest()}"

!unzip -q -n data/images.zip -d data/
!ls data | head

BASE = "https://raw.githubusercontent.com/AlexOlsen/DeepWeeds/master/labels"
for name in ["labels", "train_subset0", "val_subset0", "test_subset0"]:
    !wget -q -O data/labels/{name}.csv {BASE}/{name}.csv
!ls -la data/labels

In [ ]:
IMAGES_DIR = "data/images"   # TODO: kiểm tra đúng thư mục chứa các file .jpg sau khi giải nén (có thể là "data")
LABELS_DIR = "data/labels"

# Nơi lưu kết quả. Colab xoá /content khi ngắt phiên -> nên bật USE_DRIVE để không mất checkpoint/log.
USE_DRIVE = False
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
RUNS_DIR = "/content/drive/MyDrive/deepweeds_runs" if USE_DRIVE else "runs"
SUB_DIR = os.path.dirname(CODE_DIR)          # submissions/2A202602973_NguyenVanQuocViet

# Tham số đường dẫn dùng chung cho mọi Config. Dự đoán val của B/T nằm trong RUNS_DIR;
# chỉ chung kết + mốc (Bước 4) mới ghi vào SUB_DIR/predictions để nộp.
PATHS = dict(images_dir=IMAGES_DIR, labels_dir=LABELS_DIR, out_dir=f"{RUNS_DIR}/runs",
             pred_dir=f"{RUNS_DIR}/predictions", curves_dir=f"{RUNS_DIR}/curves")

def sync_outputs():
    """Chép ảnh biểu đồ từ RUNS_DIR sang SUB_DIR/curves (để commit nộp bài)."""
    import shutil, glob
    os.makedirs(f"{SUB_DIR}/curves", exist_ok=True)
    for f in glob.glob(f"{RUNS_DIR}/curves/*.png"):
        shutil.copy2(f, f"{SUB_DIR}/curves/")
    print("Đã chép", len(glob.glob(f"{SUB_DIR}/curves/*.png")), "ảnh vào", f"{SUB_DIR}/curves")

## Bước 0 — EDA và kiểm tra pipeline
Mục tiêu: hiểu dữ liệu, chạy các kiểm tra chia dữ liệu (README mục 2.1), kiểm tra pipeline trước khi chạy thật.

In [ ]:
# 0.1 — Đọc và kiểm tra split (README mục 2.1). check_split dừng bằng assert nếu vi phạm.
import matplotlib.pyplot as plt
from collections import Counter
from PIL import Image
import dataset as D

train_df, val_df, test_df = D.load_split(LABELS_DIR, fold=0)
split_stats = D.check_split(train_df, val_df, test_df, IMAGES_DIR)
print("\nKẾT LUẬN: giao rỗng:", all(v == 0 for v in split_stats["overlap"].values()),
      "| hợp =", split_stats["union"], "| thiếu file:", split_stats["missing"],
      "| cảnh báo tỉ lệ:", split_stats["warnings"] or "không")

In [ ]:
# 0.2 — EDA: phân bố lớp, đối chiếu Table 1 bài báo, tỉ lệ mất cân bằng, kích thước/kênh ảnh
PAPER = {"Chinee Apple": 1125, "Lantana": 1064, "Parkinsonia": 1031, "Parthenium": 1022, "Prickly Acacia": 1062,
         "Rubber Vine": 1009, "Siam Weed": 1074, "Snake Weed": 1016, "Negatives": 9106}
per_class = pd.DataFrame(split_stats["per_class"])
per_class["paper"] = pd.Series(PAPER)
per_class["khớp bài báo"] = per_class["total"] == per_class["paper"]
per_class["% tổng"] = (100 * per_class["total"] / per_class["total"].sum()).round(2)
display(per_class)
imb = per_class["total"].max() / per_class["total"].min()
print(f"Tỉ lệ lớp nhiều nhất / ít nhất = {per_class['total'].idxmax()} {per_class['total'].max()} / "
      f"{per_class['total'].idxmin()} {per_class['total'].min()} = {imb:.2f}")

ax = per_class[["train", "val", "test"]].plot.bar(stacked=True, figsize=(10, 4),
                                                  title="Phân bố lớp DeepWeeds theo tập (fold 0)")
ax.set_ylabel("số ảnh"); plt.xticks(rotation=30, ha="right"); plt.tight_layout(); plt.show()

# Kích thước và số kênh (chỉ đọc header của mọi ảnh)
all_files = pd.concat([train_df, val_df, test_df]).Filename
shapes = Counter()
for f in all_files:
    with Image.open(f"{IMAGES_DIR}/{f}") as im:
        shapes[(im.size, im.mode)] += 1
print("Kích thước (W, H) và mode:", dict(shapes))

# Thống kê màu trên 500 ảnh train (tham khảo; vẫn chuẩn hoá theo mean/std của trọng số tiền huấn luyện)
arr = np.stack([np.asarray(Image.open(f"{IMAGES_DIR}/{f}").convert("RGB"), dtype=np.float32) / 255
                for f in train_df.Filename.sample(500, random_state=0)])
print("mean RGB train ~", arr.mean((0, 1, 2)).round(3), "| std ~", arr.std((0, 1, 2)).round(3),
      "| ImageNet:", D.IMAGENET_MEAN, D.IMAGENET_STD)

In [ ]:
# 0.3 — Xem 4 ảnh mỗi lớp: cặp loài nào dễ nhầm, Negative trông ra sao
fig, axes = plt.subplots(D.NUM_CLASSES, 4, figsize=(8, 19))
for c in range(D.NUM_CLASSES):
    for j, f in enumerate(train_df[train_df.Label == c].sample(4, random_state=0).Filename):
        axes[c, j].imshow(Image.open(f"{IMAGES_DIR}/{f}")); axes[c, j].set_xticks([]); axes[c, j].set_yticks([])
    axes[c, 0].set_ylabel(D.CLASS_NAMES[c], fontsize=9)
plt.tight_layout(); plt.show()
# TODO (báo cáo): ghi nhận xét — ví dụ Chinee Apple vs Snake Weed, Parkinsonia vs Prickly Acacia.

In [ ]:
# 0.4 — Kiểm tra pipeline (GUIDE mục 1.3) trên resnet50
import math, torch.nn.functional as F
import model as M, losses as L
from train import set_seed, evaluate

set_seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
m = M.build_model("resnet50", num_classes=9, init="finetune").to(device)
cfgm = m.pretrained_cfg
print("trọng số:", m.weights_tag, "| mean/std:", cfgm["mean"], cfgm["std"])
tf_tr = D.build_transforms(True, 224, "basic", cfgm["mean"], cfgm["std"])
tf_ev = D.build_transforms(False, 224, mean=cfgm["mean"], std=cfgm["std"])

# (1) seed: hai loader cùng seed cho cùng thứ tự batch và cùng augmentation (kể cả trong worker)
def first_batch(seed):
    set_seed(seed)
    ld = D.make_loader(train_df, IMAGES_DIR, tf_tr, batch_size=16, train=True, num_workers=2, seed=seed)
    return next(iter(ld))
a, b = first_batch(0), first_batch(0)
assert a[2] == b[2] and torch.equal(a[0], b[0]), "seed không tái lập được batch đầu"
print("(1) seed tái lập thứ tự + augmentation: OK")

# (2) loss ban đầu của head mới ~ -ln(1/9) = 2.197 (trung bình trên 256 ảnh val)
val_ld = D.make_loader(val_df.sample(256, random_state=0), IMAGES_DIR, tf_ev, batch_size=64, train=False)
_, _, _, loss0 = evaluate(m, val_ld, torch.nn.CrossEntropyLoss(), device, amp=False)
print(f"(2) loss ban đầu {loss0:.3f} (kỳ vọng ~{-math.log(1/9):.3f})")

# (3) overfit một batch 32 ảnh (transform eval, không augmentation) tới loss gần 0
x, y, _ = next(iter(D.make_loader(train_df, IMAGES_DIR, tf_ev, batch_size=32, train=True, num_workers=0)))
x, y = x.to(device), y.to(device)
M.set_train_mode(m)
opt = torch.optim.AdamW(M.param_groups(m, 1e-4, 1e-3, 0.0))
for step in range(60):
    loss = F.cross_entropy(m(x), y)
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 10 == 0 or step == 59:
        print(f"(3) step {step:2d} loss {loss.item():.4f}")
assert loss.item() < 0.05, "không overfit được 1 batch -> kiểm tra code/model trước khi chạy tiếp"

# (4) ảnh sau augmentation (đã giải chuẩn hoá) + nhãn; hàng dưới: CutMix trộn cả ảnh lẫn nhãn
ds_aug = D.DeepWeedsDataset(train_df.sample(8, random_state=1), IMAGES_DIR, tf_tr)
mean, std = torch.tensor(cfgm["mean"])[:, None, None], torch.tensor(cfgm["std"])[:, None, None]
show = lambda t: (t * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()
xs = torch.stack([ds_aug[i][0] for i in range(8)]); ys = torch.tensor([ds_aug[i][1] for i in range(8)])
xm, (ya, yb, lam) = L.mix_batch(xs, ys, 1.0, "cutmix")
fig, axes = plt.subplots(2, 8, figsize=(16, 4.8))
for i in range(8):
    axes[0, i].imshow(show(xs[i])); axes[0, i].set_title(D.CLASS_NAMES[ys[i]], fontsize=8); axes[0, i].axis("off")
    axes[1, i].imshow(show(xm[i])); axes[1, i].axis("off")
    axes[1, i].set_title(f"{D.CLASS_NAMES[ya[i]][:9]} /\n{D.CLASS_NAMES[yb[i]][:9]}", fontsize=7)
fig.suptitle(f"Trên: augmentation 'basic' + nhãn. Dưới: CutMix (nhãn a / b), lam = {lam:.2f}")
plt.tight_layout(); plt.show()

# (5) train()/eval(): evaluate() phải tắt BN/Dropout; frozen phải giữ BN ở eval khi train
m.train(); evaluate(m, val_ld, None, device, amp=False)
assert not any(mod.training for mod in m.modules()), "evaluate() không gọi model.eval()"
_, _, l1, _ = evaluate(m, val_ld, None, device, amp=False)
_, _, l2, _ = evaluate(m, val_ld, None, device, amp=False)
assert np.allclose(l1, l2, atol=1e-4), "hai lần eval khác nhau -> còn ngẫu nhiên (Dropout/BN) lúc eval"
mf = M.build_model("resnet50", num_classes=9, init="frozen")
M.set_train_mode(mf)
bns = [mod for mod in mf.modules() if isinstance(mod, torch.nn.BatchNorm2d)]
assert mf.get_classifier().training and not any(bn.training for bn in bns)
print(f"(5) evaluate() -> eval: OK | frozen: head ở train, {len(bns)} BN ở eval: OK")
del m, mf, opt; torch.cuda.empty_cache()

In [ ]:
# 0.5 — Bộ test của repo (eval.py + khung) và test tự viết của code/ (focal γ=0, CutMix, param_groups, BN, ...)
!cd {REPO_DIR} && python -m unittest discover -s tests
!cd {CODE_DIR} && python -m unittest -v test_code 2>&1 | tail -30

## Bước 1 — So sánh backbone (≥ 5)
Cùng công thức nền `T00` (GUIDE.md mục 1.4), cùng seed, mỗi backbone một lần chạy. Mã thí nghiệm `B01`, `B02`, ...

In [ ]:
import json
from pathlib import Path
from train import Config, run
import benchmark as Bm

# >= 5 backbone: ResNet (mốc), ConvNeXt, 2 transformer, 2 mạng nhẹ. Cùng công thức nền T00, cùng seed 0.
BACKBONES = [
    ("B01", "resnet50"),
    ("B02", "convnext_tiny"),
    ("B03", "deit_small_patch16_224"),
    ("B04", "swin_tiny_patch4_window7_224"),
    ("B05", "efficientnet_b0"),
    ("B06", "mobilenetv3_large_100"),
]

bb_rows = []
for exp_id, bb in BACKBONES:
    summ = Path(PATHS["out_dir"]) / exp_id / "seed0" / "summary.json"
    if summ.exists():                       # đã chạy xong (vd trước khi Colab ngắt) -> bỏ qua
        s = json.loads(summ.read_text())
    else:
        s = run(Config(exp_id=exp_id, backbone=bb, desc=bb, seed=0, **PATHS))
    # độ trễ sơ bộ batch 1 (đo kỹ ở Bước 3)
    if "latency_b1_p50_ms" not in s:
        m = M.build_model(bb, num_classes=9).to(device)
        s.update({f"latency_b1_{k}_ms": v for k, v in Bm.latency_report(m, 1, 224, "fp32", device.type, iters=50).items()
                  if k in ("p50", "p95")})
        summ.write_text(json.dumps(s, indent=2)); del m
    bb_rows.append(s)

bb_df = pd.DataFrame(bb_rows)[["exp_id", "backbone", "weights_tag", "params_M", "gmacs", "best_epoch",
                               "val_macro_f1", "val_top1", "val_balanced_acc", "train_time_per_epoch_s",
                               "latency_b1_p50_ms", "latency_b1_p95_ms"]]
display(bb_df.sort_values("val_macro_f1", ascending=False))
sync_outputs()
# TODO: chọn 1-2 backbone đi tiếp, kèm lý do dựa trên số liệu (macro-F1 val, độ trễ, kích thước).

## Bước 2 — Công thức huấn luyện (≥ 3 trục)
Mỗi lần chạy chỉ khác `T00` **một yếu tố**. Mã `T01`, `T02`, ... Ghi rõ khác ở điểm nào.

In [ ]:
# TODO: các trục A-G của GUIDE.md mục 3 (khởi tạo, augmentation, loss, sampler, LR/optimizer, EMA, ...).
# TODO: thử ít nhất một kết hợp các yếu tố tốt; so Δ với nhiễu (std).

## Bước 3 — Suy luận (≥ 4 phương pháp) và độ trễ
Làm trên **val**. Không huấn luyện lại. Mã `I00` (1 view, mốc), `I01`, ...

In [ ]:
# TODO: TTA lật, multi-crop/scale, dò độ phân giải, gộp xác suất vs logit, ensemble, EMA/soup,
#       temperature scaling (T khớp trên VAL), gộp BN/FP16  (starter/inference.py)
# TODO: độ trễ p50/p95/p99 bằng starter/benchmark.py: warmup, synchronize, >= 50 lần

## Bước 4 — Chung kết: ≥ 3 seed, test một lần mỗi seed
1. Chốt cấu hình trên **val**.
2. Chạy cấu hình cuối **và mốc** (`T00` + `I00`) với cùng số seed, bật `save_test_predictions=True`.
3. Dự đoán lưu ở `predictions/<exp_id>_seed<k>_test.csv`. Rồi chạy `eval.py` (ô dưới).

In [ ]:
# TODO: chạy chung kết và mốc, mỗi seed, ví dụ:
#   for seed in (0, 1, 2):
#       run(Config(exp_id="F01", ..., seed=seed, save_test_predictions=True))
#       run(Config(exp_id="T00", seed=seed, save_test_predictions=True))
# Với suy luận (TTA/ensemble/temperature scaling), tự ghi predictions/F01_seed<k>_test.csv
# bằng eval.save_predictions(...) từ xác suất cuối cùng; thêm F01uncal_* nếu muốn chấm I4(a).

In [ ]:
# Tính chỉ số theo đúng định nghĩa của lớp (đã viết sẵn, không sửa eval.py):
!python {REPO_DIR}/eval.py score --pred "predictions/F01_seed*_test.csv" \
    --test-csv {LABELS_DIR}/test_subset0.csv --labels {LABELS_DIR}/labels.csv --tag F01 --out eval_out
!python {REPO_DIR}/eval.py score --pred "predictions/T00_seed*_test.csv" \
    --test-csv {LABELS_DIR}/test_subset0.csv --labels {LABELS_DIR}/labels.csv --tag T00 --out eval_out

In [ ]:
# Tự chấm phần I của RUBRIC (đề xuất, giảng viên xác nhận). Thêm --uncal, --final-val, --latency-p95-ms nếu có.
!python {REPO_DIR}/eval.py grade --final "predictions/F01_seed*_test.csv" \
    --baseline "predictions/T00_seed*_test.csv" \
    --test-csv {LABELS_DIR}/test_subset0.csv --labels {LABELS_DIR}/labels.csv --out eval_out

## Bước 5 — Sản phẩm
`results.xlsx` (các sheet ở GUIDE.md mục 6.1), `curves/<exp_id>_*.png`, `report.md`, `predictions/`, `code/`.

In [ ]:
# TODO: ghi results.xlsx bằng pandas.ExcelWriter(engine="openpyxl") với các sheet:
#   Backbones, Training, Inference, Final, PerClass, Latency, Summary
# TODO: kiểm tra mỗi exp_id có ảnh trong curves/ và số trong xlsx khớp kết quả của eval.py